In [ ]:
import nibabel as nib
import numpy as np

atlas = nib.load("./AAL3v1_for_SPM12/AAL3/AAL3v1.nii")

image = np.array(atlas.get_fdata(), dtype=np.int32)
affine = atlas.affine

region_names = {0: "Undefined"}
with open("./AAL3v1_for_SPM12/AAL3/AAL3v1.nii.txt", encoding="utf8", mode="r") as file:
    for line in file:
        columns = line.split()
        region_names[int(columns[0])] = columns[1]


def find_regions(coordinates: np.ndarray) -> list[str]:

    mni_coords = np.round(coordinates).astype(np.int32)  # round to ints
    atlas_coords = _convert_mni_to_atlas_space(mni_coords)
    regions = _find_regions(atlas_coords)
    
    return regions

def _convert_mni_to_atlas_space(mni_coords: np.ndarray) -> np.ndarray:
    
    extended_mni_coords = np.hstack(
        (mni_coords, np.ones((mni_coords.shape[0], 1), dtype=np.int32))
    )
    # np.linalg.solve faster than taking inverse of affine and multiplying
    atlas_coords = np.linalg.solve(affine, extended_mni_coords.T)

    return atlas_coords[:3, :].astype(np.int32).T

def _find_regions(atlas_coords: np.ndarray) -> str:

    region_ids = image[
        atlas_coords[:, 0], atlas_coords[:, 1], atlas_coords[:, 2]
    ]

    return [region_names[region_id] for region_id in region_ids]


In [ ]:
import nibabel as nib
import numpy as np
from collections import deque

atlas = nib.load("./AAL3v1_for_SPM12/AAL3/AAL3v1.nii")

image = np.array(atlas.get_fdata(), dtype=np.int32)
affine = atlas.affine

region_names = {0: "Undefined"}
with open("./AAL3/AAL3v1.nii.txt", encoding="utf8", mode="r") as file:
    for line in file:
        columns = line.split()
        region_names[int(columns[0])] = columns[1]


def find_regions(coordinates: np.ndarray) -> list[str]:

    mni_coords = np.round(coordinates).astype(np.int32)  # round to ints
    atlas_coords = _convert_mni_to_atlas_space(mni_coords)
    regions = _find_regions(atlas_coords)
    
    return regions

def _convert_mni_to_atlas_space(mni_coords: np.ndarray) -> np.ndarray:
    
    extended_mni_coords = np.hstack(
        (mni_coords, np.ones((mni_coords.shape[0], 1), dtype=np.int32))
    )
    # np.linalg.solve faster than taking inverse of affine and multiplying
    atlas_coords = np.linalg.solve(affine, extended_mni_coords.T)

    return atlas_coords[:3, :].astype(np.int32).T


def _find_regions(atlas_coords: np.ndarray) -> list[str]:
    region_ids = []
    for coord in atlas_coords:
        region_id = image[tuple(coord)]
        
        if region_id == 0:
            region_id = _find_nearest_nonzero(coord)
            
        region_ids.append(region_id) # region_names[region_id]
        
    return region_ids

def _find_nearest_nonzero(coord: np.ndarray) -> int:
    directions = np.array([
        [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1],
        [1, 1, 0], [1, -1, 0], [-1, 1, 0], [-1, -1, 0],
        [1, 0, 1], [1, 0, -1], [-1, 0, 1], [-1, 0, -1],
        [0, 1, 1], [0, 1, -1], [0, -1, 1], [0, -1, -1],
        [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]
    ])
    queue = deque([(coord, 0)])
    seen = set()
    seen.add(tuple(coord))

    while queue:
        current, dist = queue.popleft()
        if image[tuple(current)] != 0:
            return image[tuple(current)]

        for d in directions:
            neighbor = current + d
            if tuple(neighbor) not in seen and all(0 <= n < dim for n, dim in zip(neighbor, image.shape)):
                seen.add(tuple(neighbor))
                queue.append((neighbor, dist + 1))
    
    return 0  

In [ ]:
coordinates = np.array([
    [12.7, 64.7, -1.3],
    [15.7, 61.7, 6.7],
    ]
    )

regions = find_regions(coordinates)

In [ ]:
for i in regions:
    print(i)